In [1]:
import pandas as pd
import altair as alt
from scipy.stats import pearsonr
import itertools
import math

In [ ]:
# raise data limit for altair plot
alt.data_transformers.disable_max_rows()


In [2]:
summary_df = None
corr_chart = None

In [6]:
# --- load data ---
summary = pd.read_csv(summary_df)

In [79]:
# Create mutation column
summary['mutation'] = summary['wildtype'].astype(str) + summary['site'].astype(str) + summary['mutant'].astype(str)

# Select columns - we want specific pairs
columns = [
    'tufted duck MHCII binding escape',
    'entry in tufted duck MHCII cells',
    'entry in SA23 cells',
    'entry difference between noSA tufted duck MHCII and SA23 cells'
]

# Clean data
df_clean = summary.dropna(subset=columns + ['mutation', 'region']).copy()

# Selection for highlighting
mut_selection = alt.selection_point(on="mouseover", fields=['mutation'], empty=False)

# Region selection for legend
region_selection = alt.selection_point(fields=['region'], bind='legend', toggle='true')

# Create tooltips with ALL columns
all_tooltips = [
    alt.Tooltip('mutation:N', title='Mutation'),
    alt.Tooltip('region:N', title='Region'),
    alt.Tooltip('tufted duck MHCII binding escape:Q', format=".3f"),
    alt.Tooltip('entry in tufted duck MHCII cells:Q', format=".3f"),
    alt.Tooltip('entry in SA23 cells:Q', format=".3f"),
    alt.Tooltip('entry difference between noSA tufted duck MHCII and SA23 cells:Q', format=".3f")
]

# Define the specific pairs we want (first 4 plots from your image)
pairs = [
    (columns[0], columns[1]),  # binding escape vs entry in tufted duck
    (columns[0], columns[2]),  # binding escape vs entry in SA23
    (columns[0], columns[3]),  # binding escape vs entry difference
    (columns[1], columns[2]),  # entry in tufted duck vs entry in SA23
]

# Create base chart once with all parameters
base_chart = alt.Chart(df_clean).add_params(mut_selection, region_selection)

# Create scatter plots for specific pairs
scatters = []
for col_a, col_b in pairs:
    # Calculate Pearson R for this pair
    valid_data = df_clean[[col_a, col_b]].dropna()
    r, p = pearsonr(valid_data[col_a], valid_data[col_b])
    
    scatter = base_chart.transform_filter(
        f'isValid(datum["{col_a}"]) && isValid(datum["{col_b}"])'
    ).encode(
        x=alt.X(
            f"{col_a}:Q",
            title=col_a,
            scale=alt.Scale(padding=7, nice=False, zero=False),
        ),
        y=alt.Y(
            f"{col_b}:Q",
            title=col_b,
            scale=alt.Scale(padding=7, nice=False, zero=False),
        ),
    ).properties(
        width=250,
        height=250
    )
    
    # Background points
    background = scatter.mark_point(
        filled=True,
        size=30,
        color='lightgray',
        opacity=0.3,
    )
    
    # Foreground points with interactivity and color by region
    foreground = scatter.mark_point(
        filled=True,
        fillOpacity=0.6,
        stroke="black",
        strokeOpacity=1,
    ).encode(
        color=alt.Color('region:N', legend=alt.Legend(title='Region')),
        tooltip=all_tooltips,
        strokeWidth=alt.condition(mut_selection, alt.value(2), alt.value(0)),
        size=alt.condition(mut_selection, alt.value(100), alt.value(50)),
        opacity=alt.condition(region_selection, alt.value(0.6), alt.value(0.1))
    )
    
    # Add R value text - create as separate chart to avoid parameter duplication
    r_text = (
        alt.Chart(pd.DataFrame({'r_text': [f'r = {r:.2f}']}))
        .mark_text(size=14, align="left", baseline="top", color="black", fontWeight="bold")
        .encode(
            text='r_text:N',
            x=alt.value(5),
            y=alt.value(5),
        )
    )
    
    scatters.append(background + foreground + r_text)

# Arrange in 2x2 grid
chart = (
    alt.vconcat(
        alt.hconcat(scatters[0], scatters[1]),
        alt.hconcat(scatters[2], scatters[3])
    )
    .configure_axis(
        grid=False, 
        titleFontSize=11, 
        labelFontSize=10, 
        labelOverlap="greedy", 
        titleFontWeight="normal"
    )
    .configure_legend(
        titleFontSize=12,
        labelFontSize=11
    )
)

chart.save(corr_chart)
chart

alt.VConcatChart(...)